# HY2RES external validation: causal reference prediction

This notebook executes the Faisal HY2RES study plan against the locally supplied `data/external_validation_data/HY2RES_202404.csv`. It verifies the published source, audits timestamps and gaps, freezes a chronological split, compares one-step persistence with one regularized contextual autoregression, and writes the evidence package under `03_results/hy2res_external_validation/`.

**Scope boundary:** the dataset has no independently qualified leak, fault, or delay labels. This is an operational reference-prediction study only: it does not estimate incident-detection performance, physical identifiability, leak localization, or cross-plant transfer. Source timestamps are parsed as timezone-naive; no timezone is inferred.

## Frozen execution order

1. Verify Zenodo record 15166733 v1, its CC BY 4.0 license, file size, and published MD5; inspect the actual schema without guessing undocumented units.
2. Preserve all rows and assign causal segments at each OPEN/CLOSE boundary and every measured interval other than the verified 5-second step. Do not interpolate across gaps.
3. Freeze development, validation, and final chronological test periods with one-hour embargoes. Predict each supported pressure/flow channel one verified step ahead from observations available through the preceding step.
4. Select persistence or the single fixed ridge autoregression on validation only. Refit the candidate on development plus validation, then open and score the held-out test once.
5. Generate three publication figures, three main CSV tables, supporting provenance/audit files, and scoped Markdown reports only after test scoring.

In [1]:
from pathlib import Path
import hashlib
import json
import math
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.lines import Line2D
from sklearn.compose import TransformedTargetRegressor
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

project_root = Path.cwd()
while not (project_root / 'data' / 'external_validation_data' / 'HY2RES_202404.csv').is_file():
    parent = project_root.parent
    if parent == project_root:
        raise FileNotFoundError('Run this notebook from the H2-PRISM repository or one of its subfolders.')
    project_root = parent

data_path = project_root / 'data' / 'external_validation_data' / 'HY2RES_202404.csv'
results_dir = project_root / '03_results' / 'hy2res_external_validation'
figures_dir = results_dir / 'figures'
figures_dir.mkdir(parents=True, exist_ok=True)

expected_md5 = '9a3c3e3b1668596854b43bfd790d3e0e'
expected_bytes = 38621947
nominal_step_seconds = 5
embargo = pd.Timedelta(hours=1)
ridge_alpha = 10.0
minimum_regime_rows_for_figure = 30

target_units = {
    'pres_buf': 'bar',
    'pres_comp': 'bar (tank pressure; not compressor-discharge pressure)',
    'pres_pila': 'bar',
    'h2_flow': 'NL/h (generated over the preceding sample interval)',
    'h2_flow_pila': 'NL/h (fuel-cell consumption)',
}
target_channels = list(target_units)
context_power_channels = ['P_elec', 'P_comp', 'P_pila']
lag_source_channels = target_channels + context_power_channels
feature_columns = [f'{channel}_lag{lag}' for lag in (1, 2, 3) for channel in lag_source_channels]
short_units = {channel: ('NL/h' if channel.startswith('h2_flow') else 'bar') for channel in target_channels}

plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 300,
    'font.size': 9,
    'axes.titlesize': 11,
    'axes.labelsize': 9,
    'legend.fontsize': 8,
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
})
print(f'Project root: {project_root}')
print(f'Input: {data_path}')
print(f'Outputs: {results_dir}')

Project root: c:\Users\faisa\OneDrive\سطح المكتب\CURRENT_PROJECTS\H2-PRISM
Input: c:\Users\faisa\OneDrive\سطح المكتب\CURRENT_PROJECTS\H2-PRISM\data\external_validation_data\HY2RES_202404.csv
Outputs: c:\Users\faisa\OneDrive\سطح المكتب\CURRENT_PROJECTS\H2-PRISM\03_results\hy2res_external_validation


## F1 — Verify and inspect the published source

The source record identifies Zenodo version v1 (DOI `10.5281/zenodo.15166733`), collected measurements from a residential hydrogen system, and a 5-second nominal PLC interval. The local file is read once with its semicolon delimiter. The checksum and byte count are hard integrity gates; extra columns without a published definition remain explicitly undocumented and are excluded from model inputs.

In [2]:
file_md5 = hashlib.md5()
with data_path.open('rb') as source_file:
    for block in iter(lambda: source_file.read(1024 * 1024), b''):
        file_md5.update(block)
actual_md5 = file_md5.hexdigest()
actual_bytes = data_path.stat().st_size
if actual_md5 != expected_md5 or actual_bytes != expected_bytes:
    raise RuntimeError(f'SOURCE integrity check failed: bytes={actual_bytes}, MD5={actual_md5}')

raw = pd.read_csv(data_path, sep=';', low_memory=False)
published_definitions = {
    'datetime': 'PLC timestamp; published format yyyy-MM-dd hh:mm:ss; timezone unspecified',
    'Breaks': 'OPEN/CLOSE recording-period markers',
    'P_pan': 'Photovoltaic power (W)',
    'P_viv': 'Dwelling power consumption (W)',
    'P_car': 'Battery charging power (W)',
    'P_des': 'Battery discharging power (W)',
    'P_exp': 'Power exported to grid (W)',
    'P_imp': 'Power imported from grid (W)',
    'P_elec': 'Electrolyzer power consumption (W)',
    'P_comp': 'Compressor power consumption (W)',
    'P_pur': 'Purifier power consumption (W)',
    'SOC': 'Battery state of charge (%)',
    'pres_buf': 'Buffer pressure (bar)',
    'pres_comp': 'Tank pressure (bar); not identified as compressor-discharge pressure',
    'P_pila': 'Fuel-cell power output (W)',
    'pres_pila': 'Fuel-cell pressure (bar)',
    'h2_flow_pila': 'Hydrogen consumed by fuel cell (NL/h)',
    'h2_total': 'Total hydrogen generated by electrolyzer (NL)',
    'h2_flow': 'Electrolyzer hydrogen flow over the preceding sample interval (NL/h)',
    'Temperature': 'Environmental temperature (degrees C)',
}
numeric_columns = raw.select_dtypes(include=[np.number]).columns.tolist()
schema_rows = []
for column in raw.columns:
    values = raw[column]
    schema_rows.append({
        'column': column,
        'observed_dtype': str(values.dtype),
        'rows': int(len(values)),
        'missing_values': int(values.isna().sum()),
        'nonfinite_numeric_values': int((~np.isfinite(values.to_numpy(dtype=float))).sum()) if column in numeric_columns else 0,
        'published_definition': published_definitions.get(column, 'Not defined in the published record; excluded from targets and model features.'),
    })
schema_audit = pd.DataFrame(schema_rows)
schema_audit.to_csv(results_dir / 'schema_audit.csv', index=False)

channel_missingness = pd.DataFrame([
    {
        'channel': column,
        'observed_dtype': str(raw[column].dtype),
        'rows': int(len(raw)),
        'missing_values': int(raw[column].isna().sum()),
        'nonfinite_values': int((~np.isfinite(raw[column].to_numpy(dtype=float))).sum()),
        'missing_fraction': float(raw[column].isna().mean()),
    }
    for column in numeric_columns
])
channel_missingness.to_csv(results_dir / 'channel_missingness.csv', index=False)

target_channel_audit = pd.DataFrame([
    {
        'target': target,
        'unit': target_units[target],
        'source_rows_observed': int(raw[target].notna().sum()),
        'source_unique_values': int(raw[target].nunique(dropna=True)),
        'source_min': float(raw[target].min()),
        'source_max': float(raw[target].max()),
        'source_standard_deviation': float(raw[target].std(ddof=0)),
        'source_constant': bool(raw[target].nunique(dropna=True) <= 1),
    }
    for target in target_channels
])
target_channel_audit.to_csv(results_dir / 'target_channel_audit.csv', index=False)

parsed_source_time = pd.to_datetime(raw['datetime'], format='%Y-%m-%d %H:%M:%S', errors='coerce')
source_manifest = {
    'source_record': 'Zenodo record 15166733, version v1',
    'title': 'Measurements from a residential hydrogen-based energy system reported between March and April 2024',
    'doi': '10.5281/zenodo.15166733',
    'record_url': 'https://zenodo.org/records/15166733',
    'published_date': '2025-04-07',
    'license': 'CC BY 4.0',
    'source_file': 'HY2RES_202404.csv',
    'local_file': data_path.relative_to(project_root).as_posix(),
    'published_size_bytes': expected_bytes,
    'actual_size_bytes': actual_bytes,
    'published_md5': expected_md5,
    'actual_md5': actual_md5,
    'actual_sha256': hashlib.sha256(data_path.read_bytes()).hexdigest(),
    'integrity_check': 'PASS',
    'delimiter': ';',
    'rows': int(len(raw)),
    'columns': int(len(raw.columns)),
    'column_names': raw.columns.tolist(),
    'first_timestamp_as_published': str(parsed_source_time.min()),
    'last_timestamp_as_published': str(parsed_source_time.max()),
    'timestamp_timezone': 'Not supplied; retained as timezone-naive',
    'nominal_sample_interval_seconds': nominal_step_seconds,
}
with (results_dir / 'source_manifest.json').open('w', encoding='utf-8') as manifest_file:
    json.dump(source_manifest, manifest_file, indent=2, ensure_ascii=False)

print(f'Integrity PASS: {actual_bytes:,} bytes; MD5 {actual_md5}')
print(f'Observed shape: {raw.shape[0]:,} rows x {raw.shape[1]} columns')
print('Actual columns:', ', '.join(raw.columns))
print(schema_audit[['column', 'observed_dtype', 'missing_values', 'published_definition']].to_string(index=False))

Integrity PASS: 38,621,947 bytes; MD5 9a3c3e3b1668596854b43bfd790d3e0e
Observed shape: 300,768 rows x 23 columns
Actual columns: datetime, Breaks, P_pan, P_viv, P_car, P_des, P_exp, P_imp, P_elec, P_comp, P_pur, SOC, pres_buf, pres_comp, P_con, P_pila, pres_pila, h2_flow_pila, P_perdidas, h2_total, h2_inc, h2_flow, Temperature
      column observed_dtype  missing_values                                                           published_definition
    datetime         object               0      PLC timestamp; published format yyyy-MM-dd hh:mm:ss; timezone unspecified
      Breaks         object          300728                                            OPEN/CLOSE recording-period markers
       P_pan        float64               0                                                         Photovoltaic power (W)
       P_viv        float64               2                                                 Dwelling power consumption (W)
       P_car        float64               0             

## F2 — Causal time segments, gaps, and input validity

Timestamp parsing preserves the source's timezone uncertainty. Every row is retained. A segment begins on an OPEN marker, after a CLOSE marker, or whenever consecutive timestamps differ from 5 seconds (including duplicate or shorter intervals). This conservative rule prevents lag features and forecasts from bridging recording breaks or irregular intervals. Missing measurements are not interpolated; they make affected inputs or target samples ineligible.

In [3]:
analysis = raw.copy()
analysis['timestamp'] = pd.to_datetime(analysis['datetime'], format='%Y-%m-%d %H:%M:%S', errors='coerce')
if analysis['timestamp'].isna().any():
    raise ValueError(f'TIME audit failed: {int(analysis["timestamp"].isna().sum())} timestamps could not be parsed.')
if not analysis['timestamp'].is_monotonic_increasing:
    raise ValueError('TIME audit failed: source timestamps are not in nondecreasing order; no rows were reordered.')

break_markers = analysis['Breaks'].fillna('').astype(str).str.strip().str.upper()
time_deltas = analysis['timestamp'].diff().dt.total_seconds().to_numpy(dtype=float)
segment_ids = np.empty(len(analysis), dtype=np.int64)
segment_start_reasons = []
current_segment = -1
for row_index in range(len(analysis)):
    reasons = []
    if row_index == 0:
        reasons.append('first_record')
    if break_markers.iloc[row_index] == 'OPEN':
        reasons.append('Breaks_OPEN')
    if row_index > 0:
        if break_markers.iloc[row_index - 1] == 'CLOSE':
            reasons.append('after_Breaks_CLOSE')
        if not np.isfinite(time_deltas[row_index]) or time_deltas[row_index] != nominal_step_seconds:
            reasons.append('non_5_second_step')
    if reasons:
        current_segment += 1
    segment_ids[row_index] = current_segment
    segment_start_reasons.append(';'.join(reasons))
analysis['segment_id'] = segment_ids
analysis['segment_start_reason'] = segment_start_reasons

open_close_pairs = []
unmatched_open_rows = []
unmatched_close_rows = []
active_open = None
for row_index, marker in enumerate(break_markers):
    if marker == 'OPEN':
        if active_open is not None:
            unmatched_open_rows.append(active_open)
        active_open = row_index
    elif marker == 'CLOSE':
        if active_open is None:
            unmatched_close_rows.append(row_index)
        else:
            open_close_pairs.append((active_open, row_index))
            active_open = None
if active_open is not None:
    unmatched_open_rows.append(active_open)

segment_rows = []
for segment_id, group in analysis.groupby('segment_id', sort=True):
    segment_rows.append({
        'segment_id': int(segment_id),
        'start_timestamp': group['timestamp'].iloc[0],
        'end_timestamp': group['timestamp'].iloc[-1],
        'row_count': int(len(group)),
        'start_reason': group['segment_start_reason'].iloc[0],
        'open_markers': int(break_markers.iloc[group.index].eq('OPEN').sum()),
        'close_markers': int(break_markers.iloc[group.index].eq('CLOSE').sum()),
        'all_within_segment_steps_are_5s': bool(len(group) <= 1 or group['timestamp'].diff().dropna().dt.total_seconds().eq(nominal_step_seconds).all()),
    })
segment_registry = pd.DataFrame(segment_rows)
segment_registry.to_csv(results_dir / 'segment_registry.csv', index=False)

delta_series = pd.Series(time_deltas[1:])
time_audit = {
    'rows_preserved': int(len(analysis)),
    'timestamp_parse_failures': 0,
    'timestamps_monotonic_nondecreasing': True,
    'duplicate_timestamp_intervals': int(delta_series.eq(0).sum()),
    'intervals_exactly_5_seconds': int(delta_series.eq(nominal_step_seconds).sum()),
    'positive_intervals_shorter_than_5_seconds': int((delta_series.gt(0) & delta_series.lt(nominal_step_seconds)).sum()),
    'intervals_longer_than_5_seconds': int(delta_series.gt(nominal_step_seconds).sum()),
    'non_5_second_intervals': int((~delta_series.eq(nominal_step_seconds)).sum()),
    'break_open_markers': int(break_markers.eq('OPEN').sum()),
    'break_close_markers': int(break_markers.eq('CLOSE').sum()),
    'paired_open_close_markers': int(len(open_close_pairs)),
    'unmatched_open_markers': int(len(unmatched_open_rows)),
    'unmatched_close_markers': int(len(unmatched_close_rows)),
    'unrecognized_break_markers': sorted(set(break_markers[~break_markers.isin(['', 'OPEN', 'CLOSE'])].tolist())),
    'segments': int(len(segment_registry)),
    'segmentation_rule': 'New segment at OPEN, after CLOSE, and at every timestamp interval other than exactly 5 seconds; no interpolation.',
    'timezone_handling': 'Timezone-naive source values preserved; timezone not inferred.',
}
with (results_dir / 'time_audit.json').open('w', encoding='utf-8') as audit_file:
    json.dump(time_audit, audit_file, indent=2, ensure_ascii=False)

print(json.dumps(time_audit, indent=2))
print(f'Segment registry: {len(segment_registry):,} segments; all {len(analysis):,} source rows retained.')

{
  "rows_preserved": 300768,
  "timestamp_parse_failures": 0,
  "timestamps_monotonic_nondecreasing": true,
  "duplicate_timestamp_intervals": 21,
  "intervals_exactly_5_seconds": 297975,
  "positive_intervals_shorter_than_5_seconds": 32,
  "intervals_longer_than_5_seconds": 2739,
  "non_5_second_intervals": 2792,
  "break_open_markers": 20,
  "break_close_markers": 20,
  "paired_open_close_markers": 20,
  "unmatched_open_markers": 0,
  "unmatched_close_markers": 0,
  "unrecognized_break_markers": [],
  "segments": 2793,
  "segmentation_rule": "New segment at OPEN, after CLOSE, and at every timestamp interval other than exactly 5 seconds; no interpolation.",
  "timezone_handling": "Timezone-naive source values preserved; timezone not inferred."
}
Segment registry: 2,793 segments; all 300,768 source rows retained.


## F3 — Freeze targets, causal features, and chronological roles

Targets are the three publisher-defined pressure channels and two publisher-defined hydrogen-flow channels, in their recorded units. The forecast target at time *t* uses only values observed through *t − 1*. Context comprises the preceding three valid samples of the target channels and the documented equipment powers `P_elec`, `P_comp`, and `P_pila`; undocumented fields are not used. The regime tag is computed from lag-1 equipment powers, so it is available at forecast time.

Chronological roles are development (first 70%), validation (next 15%), and final test (last 15%), with one-hour embargoes after the development and validation cutoffs. The split and model-selection rule are saved before validation/model fitting. The held-out test is not used to select the model or any thresholds.

In [4]:
timestamp_ns = analysis['timestamp'].astype('int64').to_numpy()
development_cutoff = pd.Timestamp(int(np.quantile(timestamp_ns, 0.70)))
validation_cutoff = pd.Timestamp(int(np.quantile(timestamp_ns, 0.85)))
split_role = np.full(len(analysis), 'embargo', dtype=object)
split_role[analysis['timestamp'].le(development_cutoff).to_numpy()] = 'development'
validation_mask = analysis['timestamp'].gt(development_cutoff + embargo) & analysis['timestamp'].le(validation_cutoff)
split_role[validation_mask.to_numpy()] = 'validation'
split_role[analysis['timestamp'].gt(validation_cutoff + embargo).to_numpy()] = 'test'

data = analysis.copy()
data['split_role'] = split_role
for lag in (1, 2, 3):
    lagged_values = data.groupby('segment_id', sort=False)[lag_source_channels].shift(lag)
    for channel in lag_source_channels:
        data[f'{channel}_lag{lag}'] = lagged_values[channel]
    lagged_time = data.groupby('segment_id', sort=False)['timestamp'].shift(lag)
    has_lag = lagged_time.notna()
    actual_lag_seconds = (data.loc[has_lag, 'timestamp'] - lagged_time.loc[has_lag]).dt.total_seconds()
    if not actual_lag_seconds.eq(lag * nominal_step_seconds).all():
        raise AssertionError(f'Feature-availability check failed for lag {lag}; a lag crossed a discontinuity.')

feature_values = data[feature_columns].to_numpy(dtype=float)
data['feature_ready'] = np.isfinite(feature_values).all(axis=1)
power_lag_columns = [f'{channel}_lag1' for channel in context_power_channels]
power_lag_values = data[power_lag_columns].to_numpy(dtype=float)
regime_known = np.isfinite(power_lag_values).all(axis=1)
active_power = np.isfinite(power_lag_values) & (power_lag_values > 0)
active_count = active_power.sum(axis=1)
regime = np.full(len(data), 'unknown', dtype=object)
regime[regime_known & (active_count == 0)] = 'all_off'
regime[regime_known & (active_count > 1)] = 'multiple_active'
for column_index, channel in enumerate(context_power_channels):
    regime[regime_known & (active_count == 1) & active_power[:, column_index]] = {
        'P_elec': 'electrolyzer_active',
        'P_comp': 'compressor_active',
        'P_pila': 'fuel_cell_active',
    }[channel]
data['operating_regime'] = regime

role_order = ['development', 'validation', 'embargo', 'test']
validity_mask = pd.DataFrame({
    'source_row_number': np.arange(1, len(data) + 1),
    'timestamp': data['timestamp'],
    'segment_id': data['segment_id'],
    'split_role': data['split_role'],
    'causal_lags_and_predictors_finite': data['feature_ready'],
    'operating_regime_at_forecast_origin': data['operating_regime'],
})
validity_summary_rows = []
for target in target_channels:
    target_observed = np.isfinite(data[target].to_numpy(dtype=float))
    eligible_target = target_observed & data['feature_ready'].to_numpy()
    validity_mask[f'{target}_target_observed'] = target_observed
    validity_mask[f'{target}_prediction_eligible'] = eligible_target
    exclusion_reason = np.full(len(data), 'eligible', dtype=object)
    exclusion_reason[~target_observed & ~data['feature_ready'].to_numpy()] = 'target_and_causal_inputs_unavailable'
    exclusion_reason[~target_observed & data['feature_ready'].to_numpy()] = 'target_missing_or_nonfinite'
    exclusion_reason[target_observed & ~data['feature_ready'].to_numpy()] = 'causal_history_or_predictor_missing'
    validity_mask[f'{target}_exclusion_reason'] = exclusion_reason
    for role in role_order:
        role_rows = data['split_role'].eq(role).to_numpy()
        observed_count = int((role_rows & target_observed).sum())
        eligible_count = int((role_rows & eligible_target).sum())
        validity_summary_rows.append({
            'split_role': role,
            'target': target,
            'source_rows_in_role': int(role_rows.sum()),
            'target_observed_rows': observed_count,
            'causal_prediction_eligible_rows': eligible_count,
            'target_missing_rows': int((role_rows & ~target_observed).sum()),
            'causal_history_or_predictor_unavailable_rows': int((role_rows & target_observed & ~data['feature_ready'].to_numpy()).sum()),
            'validity_fraction_of_observed_targets': float(eligible_count / observed_count) if observed_count else float('nan'),
        })
validity_mask.to_csv(results_dir / 'validity_mask.csv.gz', index=False, compression='gzip')
validity_summary = pd.DataFrame(validity_summary_rows)
validity_summary.to_csv(results_dir / 'validity_summary.csv', index=False)

split_manifest_rows = []
for role in role_order:
    role_rows = data.loc[data['split_role'].eq(role), 'timestamp']
    split_manifest_rows.append({
        'split_role': role,
        'rows': int(len(role_rows)),
        'fraction_of_source_rows': float(len(role_rows) / len(data)),
        'first_timestamp': str(role_rows.min()) if len(role_rows) else '',
        'last_timestamp': str(role_rows.max()) if len(role_rows) else '',
    })
split_manifest = pd.DataFrame(split_manifest_rows)
split_manifest.to_csv(results_dir / 'split_manifest.csv', index=False)
if any(split_manifest.set_index('split_role').loc[role, 'rows'] == 0 for role in ['development', 'validation', 'test']):
    raise RuntimeError('SPLIT failed: development, validation, and test must all contain rows.')

protocol = {
    'protocol_status': 'FROZEN_BEFORE_MODEL_FIT_AND_TEST_SCORING',
    'source_record': 'Zenodo 15166733 v1; CC BY 4.0',
    'source_md5': actual_md5,
    'target_definition': 'Per-channel one-step-ahead reference prediction at the next verified 5-second sample.',
    'target_channels_and_units': target_units,
    'baseline': 'Persistence: predict each target using that channel value at t-1.',
    'single_candidate': 'Ridge(alpha=10) with standardized predictors and standardized target, separately per channel.',
    'candidate_predictors': feature_columns,
    'predictor_availability': 'Only t-1, t-2, and t-3 values within the same continuous segment; transforms fit on training rows only.',
    'equipment_context_channels': context_power_channels,
    'development_cutoff': str(development_cutoff),
    'validation_cutoff': str(validation_cutoff),
    'embargo': str(embargo),
    'split_policy': 'Chronological 70/15/15 cutoffs; one-hour embargo before validation and test; roles frozen before model fitting.',
    'model_selection': 'Select contextual Ridge only when the unweighted mean validation RMSE ratio is < 1 across channels with nonzero persistence RMSE; constant-baseline channels have undefined ratios and are excluded.',
    'test_policy': 'Final test is scored once after selection; no test-based tuning or model selection.',
    'segmentation_policy': time_audit['segmentation_rule'],
    'timezone_policy': time_audit['timezone_handling'],
    'metric_policy': 'MAE and RMSE per channel in source units; coverage and effective denominators reported; regime strata are descriptive.',
    'prohibited_claims': ['incident detection', 'false-alarm rate without incident truth', 'leak localization', 'certified D(Y)', 'cross-plant transfer'],
}
with (results_dir / 'protocol.json').open('w', encoding='utf-8') as protocol_file:
    json.dump(protocol, protocol_file, indent=2, ensure_ascii=False)

print('Frozen chronological split:')
print(split_manifest.to_string(index=False))
print(f'Feature availability verified: {len(feature_columns)} lag features; no segment-crossing lags.')
print('Protocol saved before validation fitting and final test scoring.')

Frozen chronological split:
 split_role   rows  fraction_of_source_rows     first_timestamp      last_timestamp
development 210537                 0.699998 2024-03-27 11:34:27 2024-04-09 02:34:59
 validation  44396                 0.147609 2024-04-09 03:35:05 2024-04-12 11:13:50
    embargo   1438                 0.004781 2024-04-09 02:35:04 2024-04-12 12:13:51
       test  44397                 0.147612 2024-04-12 12:13:56 2024-04-15 02:00:00
Feature availability verified: 24 lag features; no segment-crossing lags.
Protocol saved before validation fitting and final test scoring.


## F4 — Fit the baseline and one contextual extension; select on validation

Persistence is the fixed comparator. The only extension is a per-target ridge autoregression with lags 1–3 of the five targets and three documented equipment-power channels. Feature and target scalers are fitted only on development rows. The model choice is determined by the unweighted mean contextual-to-persistence validation RMSE ratio across channels with nonzero persistence RMSE; constant-baseline channels have undefined ratios and are excluded. Ties or no improvement select persistence. Validation predictions and the selection table are saved before final test access.

In [5]:
def rmse(actual, predicted):
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)
    return float(np.sqrt(np.mean((actual - predicted) ** 2))) if len(actual) else float('nan')

def fit_context_models(training_mask):
    models = {}
    training_counts = {}
    for target in target_channels:
        target_values = data[target].to_numpy(dtype=float)
        eligible = training_mask & np.isfinite(target_values)
        if int(eligible.sum()) < 2:
            raise RuntimeError(f'MODEL failed: fewer than two eligible training rows for {target}.')
        model = TransformedTargetRegressor(
            regressor=make_pipeline(StandardScaler(), Ridge(alpha=ridge_alpha)),
            transformer=StandardScaler(),
        )
        model.fit(data.loc[eligible, feature_columns].to_numpy(dtype=float), target_values[eligible])
        models[target] = model
        training_counts[target] = int(eligible.sum())
    return models, training_counts

development_rows = data['split_role'].eq('development').to_numpy() & data['feature_ready'].to_numpy()
validation_rows = data['split_role'].eq('validation').to_numpy() & data['feature_ready'].to_numpy()
if not development_rows.any() or not validation_rows.any():
    raise RuntimeError('MODEL development/validation has no rows with complete causal lag features.')
development_models, development_training_counts = fit_context_models(development_rows)

validation_records = []
validation_metric_rows = []
for target in target_channels:
    target_values = data[target].to_numpy(dtype=float)
    eligible = validation_rows & np.isfinite(target_values)
    if not eligible.any():
        raise RuntimeError(f'METRIC failed: no eligible validation targets for {target}.')
    x_validation = data.loc[eligible, feature_columns].to_numpy(dtype=float)
    actual = target_values[eligible]
    persistence_prediction = data.loc[eligible, f'{target}_lag1'].to_numpy(dtype=float)
    contextual_prediction = development_models[target].predict(x_validation)
    validation_records.append(pd.DataFrame({
        'timestamp': data.loc[eligible, 'timestamp'].to_numpy(),
        'segment_id': data.loc[eligible, 'segment_id'].to_numpy(),
        'target': target,
        'unit': target_units[target],
        'operating_regime': data.loc[eligible, 'operating_regime'].to_numpy(),
        'y_true': actual,
        'persistence_prediction': persistence_prediction,
        'contextual_prediction': contextual_prediction,
    }))
    baseline_rmse = rmse(actual, persistence_prediction)
    context_rmse = rmse(actual, contextual_prediction)
    validation_metric_rows.append({
        'target': target,
        'unit': target_units[target],
        'validation_rows': int(len(actual)),
        'persistence_mae': float(np.mean(np.abs(actual - persistence_prediction))),
        'persistence_rmse': baseline_rmse,
        'contextual_mae': float(np.mean(np.abs(actual - contextual_prediction))),
        'contextual_rmse': context_rmse,
        'contextual_to_persistence_rmse_ratio': context_rmse / baseline_rmse if baseline_rmse > 0 else float('nan'),
    })

development_validation_predictions = pd.concat(validation_records, ignore_index=True)
development_validation_predictions.to_csv(results_dir / 'development_validation_predictions.csv.gz', index=False, compression='gzip')
development_selection_metrics = pd.DataFrame(validation_metric_rows)
valid_ratios = development_selection_metrics['contextual_to_persistence_rmse_ratio'].replace([np.inf, -np.inf], np.nan).dropna()
if valid_ratios.empty:
    raise RuntimeError('MODEL selection failed: no defined validation RMSE ratios.')
macro_validation_rmse_ratio = float(valid_ratios.mean())
selection_eligible_targets = development_selection_metrics.loc[
    development_selection_metrics['contextual_to_persistence_rmse_ratio'].replace([np.inf, -np.inf], np.nan).notna(), 'target'
].tolist()
selected_model = 'contextual_ridge' if macro_validation_rmse_ratio < 1.0 else 'persistence'
development_selection_metrics['macro_validation_rmse_ratio'] = macro_validation_rmse_ratio
development_selection_metrics['selected_model'] = selected_model
development_selection_metrics.to_csv(results_dir / 'development_selection_metrics.csv', index=False)

selection_record = {
    'selection_source': 'Chronological validation only; final test not scored or inspected.',
    'development_training_rows_by_target': development_training_counts,
    'validation_scored_rows_by_target': dict(zip(development_selection_metrics['target'], development_selection_metrics['validation_rows'].astype(int))),
    'macro_mean_contextual_to_persistence_rmse_ratio': macro_validation_rmse_ratio,
    'selection_eligible_targets': selection_eligible_targets,
    'undefined_ratio_targets_excluded': development_selection_metrics.loc[~development_selection_metrics['target'].isin(selection_eligible_targets), 'target'].tolist(),
    'fixed_selection_rule': 'contextual_ridge iff unweighted mean of defined channel RMSE ratios < 1.0; otherwise persistence',
    'selected_model': selected_model,
    'validation_predictions_file': 'development_validation_predictions.csv.gz',
}
with (results_dir / 'development_model_selection.json').open('w', encoding='utf-8') as selection_file:
    json.dump(selection_record, selection_file, indent=2, ensure_ascii=False)

print(development_selection_metrics.to_string(index=False))
print(f'Frozen model choice: {selected_model}; mean validation RMSE ratio={macro_validation_rmse_ratio:.4f} across {len(selection_eligible_targets)}/{len(target_channels)} nonconstant-baseline channels.')

OSError: [Errno 22] Invalid argument: 'c:\\Users\\faisa\\OneDrive\\سطح المكتب\\CURRENT_PROJECTS\\H2-PRISM\\03_results\\hy2res_external_validation\\development_validation_predictions.csv.gz'

## F5 — Refit on development + validation; score the locked test once

The validation-based choice is now fixed. The single contextual model is refitted using only development and validation rows; no test data fit transforms or model parameters. The following cell is the sole test-scoring step. It compares both fixed references on identical eligible test rows and reports original-unit errors, observed-target denominators, reference coverage, and segment counts. There are no incident labels, so false-alarm rates are not calculated.

In [ ]:
development_validation_rows = data['split_role'].isin(['development', 'validation']).to_numpy() & data['feature_ready'].to_numpy()
final_context_models, final_training_counts = fit_context_models(development_validation_rows)
test_rows = data['split_role'].eq('test').to_numpy()
if not test_rows.any():
    raise RuntimeError('METRIC failed: frozen test split is empty.')

test_prediction_parts = []
test_observed_counts = {}
test_valid_counts = {}
test_target_summaries = {}
for target in target_channels:
    target_values = data[target].to_numpy(dtype=float)
    target_observed = test_rows & np.isfinite(target_values)
    eligible = target_observed & data['feature_ready'].to_numpy()
    test_observed_counts[target] = int(target_observed.sum())
    test_valid_counts[target] = int(eligible.sum())
    observed_values = target_values[target_observed]
    if not len(observed_values):
        raise RuntimeError(f'METRIC failed: no observed test target values for {target}.')
    test_target_summaries[target] = {
        'unique_values': int(np.unique(observed_values).size),
        'minimum': float(np.min(observed_values)),
        'maximum': float(np.max(observed_values)),
        'constant_in_test': bool(np.unique(observed_values).size <= 1),
    }
    if not eligible.any():
        raise RuntimeError(f'METRIC failed: no eligible test predictions for {target}.')
    x_test = data.loc[eligible, feature_columns].to_numpy(dtype=float)
    actual = target_values[eligible]
    persistence_prediction = data.loc[eligible, f'{target}_lag1'].to_numpy(dtype=float)
    contextual_prediction = final_context_models[target].predict(x_test)
    chosen_prediction = contextual_prediction if selected_model == 'contextual_ridge' else persistence_prediction
    test_prediction_parts.append(pd.DataFrame({
        'timestamp': data.loc[eligible, 'timestamp'].to_numpy(),
        'segment_id': data.loc[eligible, 'segment_id'].to_numpy(),
        'target': target,
        'unit': target_units[target],
        'operating_regime': data.loc[eligible, 'operating_regime'].to_numpy(),
        'y_true': actual,
        'persistence_prediction': persistence_prediction,
        'contextual_prediction': contextual_prediction,
        'selected_model': selected_model,
        'selected_prediction': chosen_prediction,
    }))

test_predictions = pd.concat(test_prediction_parts, ignore_index=True)
test_predictions.to_csv(results_dir / 'test_predictions.csv.gz', index=False, compression='gzip')
test_split_row_count = int(test_rows.sum())
metric_rows = []
for target in target_channels:
    target_observed = data.loc[test_rows & np.isfinite(data[target].to_numpy(dtype=float)), ['operating_regime']]
    target_predictions = test_predictions.loc[test_predictions['target'].eq(target)]
    regime_values = ['overall'] + sorted(target_observed['operating_regime'].dropna().unique().tolist())
    for regime_name in regime_values:
        if regime_name == 'overall':
            denominator = int(len(target_observed))
            observed_regime = target_predictions
        else:
            denominator = int(target_observed['operating_regime'].eq(regime_name).sum())
            observed_regime = target_predictions.loc[target_predictions['operating_regime'].eq(regime_name)]
        for method, prediction_column in [
            ('persistence', 'persistence_prediction'),
            ('contextual_ridge', 'contextual_prediction'),
        ]:
            actual = observed_regime['y_true'].to_numpy(dtype=float)
            predicted = observed_regime[prediction_column].to_numpy(dtype=float)
            scored_count = int(len(actual))
            if scored_count == 0:
                support_status = 'no eligible rows'
            elif regime_name == 'overall' and test_target_summaries[target]['constant_in_test']:
                support_status = 'constant target in test; errors are uninformative'
            elif scored_count < minimum_regime_rows_for_figure and regime_name != 'overall':
                support_status = 'low support (<30 scored rows); descriptive only'
            else:
                support_status = 'descriptive; not an incident-performance estimate'
            metric_rows.append({
                'target': target,
                'unit': target_units[target],
                'operating_regime': regime_name,
                'method': method,
                'selected_for_reporting': bool(method == selected_model),
                'test_split_rows': test_split_row_count,
                'observed_target_rows': denominator,
                'scored_rows': scored_count,
                'segments_scored': int(observed_regime['segment_id'].nunique()),
                'test_target_unique_values': test_target_summaries[target]['unique_values'],
                'test_target_min': test_target_summaries[target]['minimum'],
                'test_target_max': test_target_summaries[target]['maximum'],
                'test_target_constant': test_target_summaries[target]['constant_in_test'],
                'reference_coverage': float(scored_count / denominator) if denominator else float('nan'),
                'mae': float(np.mean(np.abs(actual - predicted))) if scored_count else float('nan'),
                'rmse': rmse(actual, predicted),
                'support_status': support_status,
            })

per_channel_metrics = pd.DataFrame(metric_rows)
per_channel_metrics.to_csv(results_dir / 'per_channel_reference_metrics.csv', index=False)
overall_metrics = per_channel_metrics.loc[per_channel_metrics['operating_regime'].eq('overall')]
test_summary = {
    'test_opened_for_scoring_once': True,
    'test_first_timestamp': str(data.loc[test_rows, 'timestamp'].min()),
    'test_last_timestamp': str(data.loc[test_rows, 'timestamp'].max()),
    'test_split_rows': test_split_row_count,
    'selected_model_from_validation': selected_model,
    'context_models_fit_rows_by_target': final_training_counts,
    'test_eligible_rows_by_target': test_valid_counts,
    'test_observed_target_rows_by_target': test_observed_counts,
    'test_target_variation': test_target_summaries,
    'overall_metrics': overall_metrics[['target', 'method', 'scored_rows', 'segments_scored', 'test_target_unique_values', 'test_target_constant', 'reference_coverage', 'mae', 'rmse']].to_dict(orient='records'),
    'incident_labels_available': False,
    'false_alarm_rate_reported': False,
}
with (results_dir / 'metrics.json').open('w', encoding='utf-8') as metrics_file:
    json.dump(test_summary, metrics_file, indent=2, ensure_ascii=False, allow_nan=False)

print('Final held-out test metrics (original units):')
print(overall_metrics[['target', 'unit', 'method', 'scored_rows', 'segments_scored', 'reference_coverage', 'mae', 'rmse']].to_string(index=False))
print('Regime-specific metrics are in per_channel_reference_metrics.csv; low-support strata are explicitly flagged.')

## F6 — Figures, three main tables, and scoped reporting

Figures are built from the audited source and the already-scored fixed test interval. The test trace interval is selected chronologically by segment length, not by observed errors. All three figures are saved as PNG, SVG, and PDF. Three main tables cover data/splits, per-channel reference metrics (including supported descriptive operating regimes), and limitations/claims. Markdown reports are written last, after results are available.

In [ ]:
def save_figure(figure, stem):
    for extension in ['png', 'svg', 'pdf']:
        figure.savefig(figures_dir / f'{stem}.{extension}', dpi=300, bbox_inches='tight')

def display_unit(target):
    return short_units[target]

# Figure 1: complete operational timeline, with line breaks at every audited segment boundary.
pressure_channels = ['pres_buf', 'pres_comp', 'pres_pila']
flow_channels = ['h2_flow', 'h2_flow_pila']
power_channels = ['P_elec', 'P_comp', 'P_pila']
telemetry_columns = pressure_channels + flow_channels + power_channels
telemetry = data.set_index('timestamp')[telemetry_columns].copy()
segment_starts = np.flatnonzero(data['segment_id'].diff().fillna(0).ne(0).to_numpy())
if len(segment_starts):
    telemetry.iloc[segment_starts, :] = np.nan
figure1, axes1 = plt.subplots(3, 1, figsize=(15, 10), sharex=True, constrained_layout=True)
for channel in pressure_channels:
    axes1[0].plot(telemetry.index, telemetry[channel], linewidth=0.65, label=channel)
axes1[0].set_ylabel('Pressure (bar)')
axes1[0].set_title('HY2RES operational record: pressures, hydrogen flows, and equipment power')
axes1[0].legend(ncol=3, loc='upper left')
for channel in flow_channels:
    label = 'h2_flow generation over preceding interval' if channel == 'h2_flow' else 'h2_flow_pila fuel-cell consumption'
    axes1[1].plot(telemetry.index, telemetry[channel], linewidth=0.65, label=label)
axes1[1].set_ylabel('Hydrogen flow (NL/h)')
axes1[1].legend(ncol=2, loc='upper left')
for channel in power_channels:
    axes1[2].plot(telemetry.index, telemetry[channel], linewidth=0.65, label=channel)
axes1[2].set_ylabel('Equipment power (W)')
axes1[2].set_xlabel('Source timestamp (timezone unspecified)')
axes1[2].legend(ncol=3, loc='upper left')
for row_index in np.flatnonzero(break_markers.eq('OPEN').to_numpy()):
    for axis in axes1:
        axis.axvline(data['timestamp'].iloc[row_index], color='tab:green', linewidth=0.45, alpha=0.34)
for row_index in np.flatnonzero(break_markers.eq('CLOSE').to_numpy()):
    for axis in axes1:
        axis.axvline(data['timestamp'].iloc[row_index], color='tab:red', linewidth=0.45, alpha=0.34)
pressure_handles, pressure_labels = axes1[0].get_legend_handles_labels()
boundary_handles = [
    Line2D([], [], color='tab:green', linewidth=1.0, label='Breaks OPEN'),
    Line2D([], [], color='tab:red', linewidth=1.0, label='Breaks CLOSE'),
    Line2D([], [], color='black', linewidth=1.0, linestyle=':', label='Gap >60 s'),
]
axes1[0].legend(pressure_handles + boundary_handles, pressure_labels + [handle.get_label() for handle in boundary_handles], ncol=3, loc='upper left')
long_gap_rows = np.flatnonzero(time_deltas > 60)
for row_index in long_gap_rows:
    for axis in axes1:
        axis.axvline(data['timestamp'].iloc[row_index], color='black', linewidth=0.8, linestyle=':', alpha=0.65)
axes1[0].text(0.995, 0.06, f'{len(segment_registry):,} causal segments; {time_audit["non_5_second_intervals"]:,} non-5-second intervals', transform=axes1[0].transAxes, ha='right', va='bottom', fontsize=8)
axes1[2].xaxis.set_major_locator(mdates.AutoDateLocator(minticks=6, maxticks=12))
axes1[2].xaxis.set_major_formatter(mdates.ConciseDateFormatter(axes1[2].xaxis.get_major_locator()))
save_figure(figure1, 'F1_hy2res_operational_timeline')
plt.close(figure1)

# Figure 2: first sufficiently long held-out test segment, chosen by time/length only.
test_segment_sizes = data.loc[data['split_role'].eq('test')].groupby('segment_id').size()
qualifying_segments = test_segment_sizes[test_segment_sizes.ge(120)].index.tolist()
if qualifying_segments:
    selected_trace_segment = min(qualifying_segments, key=lambda segment: data.loc[data['segment_id'].eq(segment), 'timestamp'].min())
else:
    if test_segment_sizes.empty:
        raise RuntimeError('FIGURE failed: no test segments are available for a reference trace.')
    selected_trace_segment = int(test_segment_sizes.idxmax())
trace_predictions = test_predictions.loc[test_predictions['segment_id'].eq(selected_trace_segment)].copy()
if trace_predictions.empty:
    raise RuntimeError('FIGURE failed: the selected test interval has no eligible reference predictions.')
trace_start = trace_predictions['timestamp'].min()
trace_end = trace_predictions['timestamp'].max()
figure2, axes2 = plt.subplots(len(target_channels), 1, figsize=(14, 14), sharex=True, constrained_layout=True)
for axis, target in zip(axes2, target_channels):
    trace = trace_predictions.loc[trace_predictions['target'].eq(target)].sort_values('timestamp').head(1200)
    axis.plot(trace['timestamp'], trace['y_true'], color='black', linewidth=1.0, label='Observed')
    axis.plot(trace['timestamp'], trace['persistence_prediction'], color='tab:orange', linewidth=0.8, alpha=0.9, label='Persistence')
    axis.plot(trace['timestamp'], trace['contextual_prediction'], color='tab:blue', linewidth=0.8, alpha=0.9, label='Contextual ridge')
    target_label = f'{target}\n[{display_unit(target)}]'
    if bool(target_channel_audit.loc[target_channel_audit['target'].eq(target), 'source_constant'].iloc[0]):
        target_label += '\n(constant in source)'
    axis.set_ylabel(target_label)
    axis.grid(alpha=0.2)
axes2[0].set_title(f'One-step reference predictions on a fixed test interval: segment {selected_trace_segment}')
axes2[0].legend(ncol=3, loc='upper left')
axes2[-1].set_xlabel('Source timestamp (timezone unspecified)')
axes2[-1].xaxis.set_major_locator(mdates.AutoDateLocator(minticks=4, maxticks=8))
axes2[-1].xaxis.set_major_formatter(mdates.ConciseDateFormatter(axes2[-1].xaxis.get_major_locator()))
save_figure(figure2, 'F2_hy2res_test_reference_traces')
plt.close(figure2)

# Figure 3: unit-safe relative errors, original-row coverage, and descriptive regimes.
figure3, axes3 = plt.subplots(1, 3, figsize=(17, 5.8), constrained_layout=True)
target_positions = np.arange(len(target_channels))
overall = per_channel_metrics.loc[per_channel_metrics['operating_regime'].eq('overall')]
relative_context = []
coverage_values = []
for target in target_channels:
    rows = overall.loc[overall['target'].eq(target)].set_index('method')
    baseline_rmse = float(rows.loc['persistence', 'rmse'])
    context_rmse = float(rows.loc['contextual_ridge', 'rmse'])
    relative_context.append(context_rmse / baseline_rmse if baseline_rmse > 0 else np.nan)
    coverage_values.append(float(rows.loc['persistence', 'reference_coverage']))
informative_positions = [position for position, target in enumerate(target_channels) if not test_target_summaries[target]['constant_in_test']]
constant_positions = [position for position, target in enumerate(target_channels) if test_target_summaries[target]['constant_in_test']]
for position in informative_positions:
    axes3[0].plot([position, position], [1.0, relative_context[position]], color='0.72', linewidth=1.2, zorder=1)
axes3[0].scatter(informative_positions, np.ones(len(informative_positions)), marker='o', color='tab:orange', label='Persistence', zorder=3)
axes3[0].scatter(informative_positions, [relative_context[position] for position in informative_positions], marker='^', color='tab:blue', label='Contextual ridge', zorder=3)
axes3[0].scatter(constant_positions, np.ones(len(constant_positions)), marker='x', color='0.35', label='Constant target; uninformative', zorder=4)
axes3[0].axhline(1.0, color='black', linewidth=0.8, linestyle='--')
axes3[0].set_xticks(target_positions, [f'{target}\n[{display_unit(target)}]' for target in target_channels], rotation=20, ha='right')
axes3[0].set_ylabel('RMSE relative to persistence (lower is better)')
axes3[0].set_title('Overall test error by channel')
axes3[0].legend(loc='best')
axes3[0].grid(axis='y', alpha=0.2)
axes3[1].bar(target_positions, coverage_values, color='tab:green', alpha=0.8)
axes3[1].set_xticks(target_positions, [f'{target}\n[{display_unit(target)}]' for target in target_channels], rotation=20, ha='right')
axes3[1].set_ylim(0, 1.05)
axes3[1].set_ylabel('Reference coverage among observed test targets')
axes3[1].set_title('Valid-sample coverage')
axes3[1].grid(axis='y', alpha=0.2)

regime_metrics = per_channel_metrics.loc[~per_channel_metrics['operating_regime'].eq('overall')]
regime_names = [name for name in ['all_off', 'electrolyzer_active', 'compressor_active', 'fuel_cell_active', 'multiple_active', 'unknown'] if name in regime_metrics['operating_regime'].unique()]
regime_positions = {name: position for position, name in enumerate(regime_names)}
for target in target_channels:
    points = []
    for regime_name in regime_names:
        group = regime_metrics.loc[regime_metrics['target'].eq(target) & regime_metrics['operating_regime'].eq(regime_name)].set_index('method')
        if set(['persistence', 'contextual_ridge']).issubset(group.index):
            baseline_row = group.loc['persistence']
            context_row = group.loc['contextual_ridge']
            if int(context_row['scored_rows']) >= minimum_regime_rows_for_figure and float(baseline_row['rmse']) > 0:
                points.append((regime_positions[regime_name], float(context_row['rmse']) / float(baseline_row['rmse']), int(context_row['scored_rows'])))
    if points:
        axes3[2].plot([point[0] for point in points], [point[1] for point in points], marker='o', linewidth=1.0, label=target)
axes3[2].axhline(1.0, color='black', linewidth=0.8, linestyle='--')
axes3[2].set_xticks(list(regime_positions.values()), list(regime_positions.keys()), rotation=30, ha='right')
axes3[2].set_ylabel('Contextual RMSE / persistence RMSE')
axes3[2].set_title(f'Descriptive operating regimes (n >= {minimum_regime_rows_for_figure})')
axes3[2].grid(axis='y', alpha=0.2)
if axes3[2].get_legend_handles_labels()[0]:
    axes3[2].legend(loc='best')
else:
    axes3[2].text(0.5, 0.5, 'No regime stratum met the prespecified display count.', transform=axes3[2].transAxes, ha='center', va='center')
figure3.suptitle('Held-out reference error and availability; regime splits are descriptive only', fontsize=12)
save_figure(figure3, 'F3_hy2res_error_coverage_regimes')
plt.close(figure3)

# Three main evidence tables.
split_summary = split_manifest.copy()
split_summary['nominal_role'] = split_summary['split_role'].map({
    'development': 'Model fitting',
    'validation': 'Development-only model selection',
    'embargo': 'Excluded from fit and scoring',
    'test': 'Single final evaluation',
})
split_summary.to_csv(results_dir / 'data_and_splits.csv', index=False)
per_channel_metrics.to_csv(results_dir / 'per_channel_reference_metrics.csv', index=False)
claims_table = pd.DataFrame([
    {'claim_or_limit': 'Reference prediction', 'status': 'Evaluated', 'evidence': 'One-step pressure/flow MAE and RMSE against persistence on the fixed chronological test; see per_channel_reference_metrics.csv.'},
    {'claim_or_limit': 'Reference availability', 'status': 'Measured', 'evidence': 'Valid prediction counts, observed-target denominators, coverage, and segment counts are reported per channel.'},
    {'claim_or_limit': 'Incident detection or false-alarm rate', 'status': 'Not evaluated', 'evidence': 'No independently qualified incident labels are present; no detection accuracy or false-alarm rate is claimed.'},
    {'claim_or_limit': 'Leak/fault/delay classification, D(Y), or localization', 'status': 'Out of scope', 'evidence': 'No incident labels or evidence supporting these claims; this notebook does not train an incident classifier.'},
    {'claim_or_limit': 'Cross-plant or pipeline transfer', 'status': 'Not established', 'evidence': 'One residential pilot-system record; results do not establish independent-plant generalization.'},
    {'claim_or_limit': 'Timezone and gap interpretation', 'status': 'Limited', 'evidence': 'Timezone is unspecified; non-5-second intervals and OPEN/CLOSE boundaries conservatively split causal segments without interpolation.'},
    {'claim_or_limit': 'Units and flow interpretation', 'status': 'Source-bounded', 'evidence': 'Published units are retained; h2_flow is a preceding-interval rate, pres_comp is tank pressure, and undocumented columns are excluded.'},
    {'claim_or_limit': 'Targets with no observed variation', 'status': 'No predictive evidence', 'evidence': 'pres_pila and h2_flow_pila are constant at zero across the source; their zero errors are uninformative and excluded from validation selection ratios.'},
])
claims_table.to_csv(results_dir / 'limitations_and_claims.csv', index=False)

figure_files = {
    'timeline': 'F1_hy2res_operational_timeline',
    'test_traces': 'F2_hy2res_test_reference_traces',
    'error_coverage_regimes': 'F3_hy2res_error_coverage_regimes',
}
data_split_text = '\n'.join(
    f"| {row.split_role} | {int(row.rows):,} | {row.fraction_of_source_rows:.1%} | {row.first_timestamp or '—'} | {row.last_timestamp or '—'} |"
    for row in split_summary.itertuples(index=False)
)
target_audit_by_name = target_channel_audit.set_index('target')
metric_text = '\n'.join(
    f"| {row.target} | {row.method} | {int(row.scored_rows):,} | {'constant' if row.test_target_constant else f'{int(row.test_target_unique_values):,} unique'} | {row.reference_coverage:.1%} | {row.mae:.6g} | {row.rmse:.6g} |"
    for row in overall_metrics.itertuples(index=False)
)
claim_text = '\n'.join(
    f'| {row.claim_or_limit} | {row.status} | {row.evidence} |'
    for row in claims_table.itertuples(index=False)
)
channel_summary_rows = []
for target in target_channels:
    target_overall = overall_metrics.loc[overall_metrics['target'].eq(target)].set_index('method')
    if bool(target_audit_by_name.loc[target, 'source_constant']):
        channel_summary_rows.append(f"- **{target}** ({target_units[target]}): constant at {target_audit_by_name.loc[target, 'source_min']:g} across the source; zero test error is uninformative and does not demonstrate predictive skill.")
    else:
        persistence_row = target_overall.loc['persistence']
        contextual_row = target_overall.loc['contextual_ridge']
        mae_note = '; contextual MAE is higher' if contextual_row['mae'] > persistence_row['mae'] else ''
        channel_summary_rows.append(f"- **{target}** ({target_units[target]}): persistence RMSE {persistence_row['rmse']:.6g}; contextual ridge RMSE {contextual_row['rmse']:.6g}{mae_note}.")
channel_summary = '\n'.join(channel_summary_rows)
selection_percent = (1.0 - macro_validation_rmse_ratio) * 100.0
selection_sentence = f'Validation selected **{selected_model}** (mean contextual-to-persistence RMSE ratio {macro_validation_rmse_ratio:.4f} across {len(selection_eligible_targets)}/{len(target_channels)} channels with nonzero persistence RMSE; {selection_percent:+.2f}% reduction by the selection statistic). Other channels had undefined ratios because persistence RMSE was zero.'
long_gap_count = int((pd.Series(time_deltas[1:]) > 60).sum())

source_lines = [
    '# Source and provenance',
    '',
    'Dataset: HY2RES_202404.csv, Zenodo record 15166733, version v1, DOI [10.5281/zenodo.15166733](https://doi.org/10.5281/zenodo.15166733).',
    'Dataset authors: Laura Rodriguez De Lope, Ramón Agüero, Luis Díez, Inmaculada Ortiz, Alfredo Ortiz Sainz de Aja, and Víctor Manuel Maestre Muñoz. Publisher license: CC BY 4.0.',
    'Reuse attribution: Rodriguez De Lope et al., HY2RES_202404.csv, Zenodo 15166733, version v1, DOI 10.5281/zenodo.15166733, CC BY 4.0.',
    f'Published file size: {actual_bytes:,} bytes. Published and verified MD5: `{actual_md5}`.',
    f'Local input: `{data_path.relative_to(project_root).as_posix()}`. Rows and original channel values were not rewritten.',
    f'Observed shape: {len(raw):,} rows x {len(raw.columns)} columns; timestamp span {parsed_source_time.min()} to {parsed_source_time.max()}.',
    'The source does not specify a timezone; timestamps are retained as naive local-clock values. See `source_manifest.json` and `schema_audit.csv`.',
]
(results_dir / 'SOURCE.md').write_text('\n'.join(source_lines) + '\n', encoding='utf-8')

audit_lines = [
    '# Data and time audit',
    '',
    f"- Parsed rows/columns: {len(raw):,}/{len(raw.columns)}; timestamps are nondecreasing: {analysis['timestamp'].is_monotonic_increasing}.",
    f"- Duplicate timestamp intervals: {time_audit['duplicate_timestamp_intervals']:,}; exact 5-second intervals: {time_audit['intervals_exactly_5_seconds']:,}; all other intervals: {time_audit['non_5_second_intervals']:,}.",
    f"- `Breaks` markers: {time_audit['break_open_markers']} OPEN and {time_audit['break_close_markers']} CLOSE; paired: {time_audit['paired_open_close_markers']}; unmatched OPEN/CLOSE: {time_audit['unmatched_open_markers']}/{time_audit['unmatched_close_markers']}.",
    f'- Segmentation produced {len(segment_registry):,} segments; {long_gap_count:,} measured intervals exceeded 60 seconds. Every row is retained, no gap is interpolated, and every non-5-second step begins a new segment.',
    '- Channel missingness and documented/undocumented schema are listed in `channel_missingness.csv` and `schema_audit.csv`. Undocumented columns are excluded from prediction.',
    '- The row-level validity mask and role/channel eligibility counts are saved as `validity_mask.csv.gz` and `validity_summary.csv`. A prediction is eligible only when its target and all lagged predictors are finite and all lags remain in the same segment.',
    '- Target value ranges and variation are audited in `target_channel_audit.csv`; pres_pila and h2_flow_pila are constant zero throughout this source, so their zero-error metrics are uninformative.',
    '- Coverage denominators are observed target rows in the applicable test period/regime.',
]
(results_dir / 'AUDIT.md').write_text('\n'.join(audit_lines) + '\n', encoding='utf-8')

protocol_lines = [
    '# Frozen prediction protocol',
    '',
    '- Target: each publisher-defined pressure/flow channel at the next verified 5-second sample, using measurements available through the preceding sample only.',
    '- Persistence baseline: last available value of the target channel.',
    '- Single candidate: per-channel Ridge regression (alpha 10), standardized features and target; lag 1–3 target pressures/flows and documented equipment powers.',
    '- Chronological roles and embargo are recorded in `split_manifest.csv`; the complete protocol and cutoffs are in `protocol.json`.',
    f'- Model selection used validation only: {selection_sentence}',
    '- Final test transforms/model were fitted without test rows; the test was scored once. Operating-regime results are descriptive and use equipment-power state at t-1.',
    '- No predictions bridge OPEN/CLOSE markers, duplicate timestamps, or any interval other than 5 seconds. Missing values are not imputed.',
]
(results_dir / 'PROTOCOL.md').write_text('\n'.join(protocol_lines) + '\n', encoding='utf-8')

results_lines = [
    '# HY2RES external reference-prediction results',
    '',
    f'**Model choice:** {selected_model}. {selection_sentence}',
    f"**Held-out period:** {test_summary['test_first_timestamp']} through {test_summary['test_last_timestamp']} (timezone unspecified); {test_split_row_count:,} source rows in the fixed test partition.",
    '',
    '## Per-channel held-out performance',
    '',
    '| Target | Method | Scored rows | Test target variation | Reference coverage | MAE | RMSE |',
    '|---|---|---:|---|---:|---:|---:|',
    metric_text,
    '',
    'Metrics are in the channel units shown in `per_channel_reference_metrics.csv`. Coverage is the eligible reference count divided by observed target rows, with effective denominators and segment counts included in that CSV. The table contains both methods on the same eligible test rows. Constant targets are explicitly flagged because zero error on an inactive channel is not evidence of predictive skill.',
    '',
    '### Channel-level comparison',
    '',
    channel_summary,
    '',
    '## Figures',
    '',
    '### F1 — Audited operational timeline and context',
    '',
    '![Pressures, interval hydrogen flows, and equipment powers with recording boundaries and long gaps](figures/F1_hy2res_operational_timeline.png)',
    '',
    '### F2 — Persistence and contextual reference traces on a chronological test segment',
    '',
    f'Interval shown: segment {selected_trace_segment}, {trace_start} to {trace_end}. It was chosen by chronological position and segment length, not by test error.',
    '',
    '![Observed test values and both fixed one-step reference predictions](figures/F2_hy2res_test_reference_traces.png)',
    '',
    '### F3 — Error, reference coverage, and descriptive regimes',
    '',
    '![Unit-safe relative error, per-channel coverage, and operating-regime comparisons](figures/F3_hy2res_error_coverage_regimes.png)',
    '',
    '## Main tables',
    '',
    '- `data_and_splits.csv` — source rows, chronological roles, embargo, time range, and split proportions.',
    '- `validity_summary.csv` and `validity_mask.csv.gz` — per-role, per-channel eligible-sample counts and the row-level prediction mask/exclusion reason.',
    '- `target_channel_audit.csv` — source-wide target ranges and variation; `per_channel_reference_metrics.csv` — original-unit MAE/RMSE, test variation, coverage denominators, segment counts, and regime strata.',
    '- `limitations_and_claims.csv` — explicit evidence boundary and permitted interpretation.',
    '',
    '## Limitations and claim boundary',
    '',
    '| Claim or limit | Status | Evidence / boundary |',
    '|---|---|---|',
    claim_text,
    '',
    'No alarm threshold or alert policy was specified, so no alert burden is reported. The result estimates within-record chronological reference prediction on one pilot system; it is not an incident-detection or independent-plant validation study.',
    '',
    'See [`SOURCE.md`](SOURCE.md), [`AUDIT.md`](AUDIT.md), [`PROTOCOL.md`](PROTOCOL.md), `metrics.json`, and the saved test predictions for provenance and denominators.',
]
(results_dir / 'RESULTS.md').write_text('\n'.join(results_lines) + '\n', encoding='utf-8')

readme_lines = [
    '# HY2RES external validation outputs',
    '',
    'All files in this folder are generated by `notebooks/FAISAL_PIPELINE_WALKTHROUGH.ipynb`; source code remains in that notebook. The supplied CSV remains unchanged in `data/external_validation_data/`.',
    '',
    'From the repository root, execute the notebook with a Python environment containing `requirements.txt` and Jupyter:',
    '',
    '```bash',
    'jupyter nbconvert --to notebook --execute notebooks/FAISAL_PIPELINE_WALKTHROUGH.ipynb --output HY2RES_validation_executed.ipynb --output-dir 03_results/hy2res_external_validation',
    '```',
    '',
    'Start with [`RESULTS.md`](RESULTS.md); source facts, audit, frozen protocol, machine-readable metrics, and figures are linked there.',
]
(results_dir / 'README.md').write_text('\n'.join(readme_lines) + '\n', encoding='utf-8')

print(f'Generated three figures in PNG/SVG/PDF under {figures_dir}')
print('Generated main tables: data_and_splits.csv, per_channel_reference_metrics.csv, limitations_and_claims.csv')
print('Generated SOURCE.md, AUDIT.md, PROTOCOL.md, RESULTS.md, and README.md after scoring the final test.')
print('Results overview:')
print('\n'.join(results_lines[:18]))